# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Mã chính nằm trong gói `lab/` (`labels.py`, `vlm.py`, `run.py`, `viz.py`, `reward.py`, `grpo.py`).
Notebook gọi các lệnh `python -m lab.run ...` trong **tiến trình riêng** để mỗi lần chạy giải phóng GPU.
Mọi adapter/dự đoán được ghi vào Google Drive (`OUT`), và các lệnh đều **chạy lại được** sau khi mất session
(`--resume` cho train, predict bỏ qua ảnh đã làm).

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
import os
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/vietvuivui/K4-Track-4-Day10-VisionLanguageModel.git'
if not os.path.exists('/content/lab10'):
    !git clone {REPO_URL} /content/lab10
%cd /content/lab10
!git pull -q                                   # lấy bản code mới nhất nếu repo đã có sẵn
OUT = '/content/drive/MyDrive/lab10_runs'      # adapter, dự đoán, run.json (sống sót khi mất session)
os.makedirs(OUT, exist_ok=True)

In [ ]:
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_API_TOKEN, dạng KGAT_...). KHÔNG viết khoá vào notebook.
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
!pip -q install -U kaggle                      # token KGAT_ cần bản kaggle CLI mới

In [ ]:
!python -m pytest -q            # test của đề
!python -m pytest -q lab/tests  # test của mình (nhãn, config, reward E5)

## 1. Dữ liệu: tải và chia split

In [ ]:
if not os.path.exists('data/splits/meta.json'):
    !python scripts/prepare_data.py --kaggle

In [ ]:
# Thống kê split (câu 6) + kiểm tra không phiên quay nào nằm ở hai split
import json
from collections import Counter
from scripts.labcommon import read_jsonl, CLASSES

meta = json.load(open('data/splits/meta.json'))
print('dataset_version:', meta['dataset_version'])
splits = {s: read_jsonl(f'data/splits/{s}.jsonl') for s in ('train', 'val', 'test')}
sessions = {}
for s, rows in splits.items():
    boxes = Counter(o['label'] for r in rows for o in r['objects'])
    sess = {r['session'] for r in rows}
    sessions[s] = sess
    print(f"{s:5s} {len(rows):5d} ảnh  {len(sess):3d} phiên  " + '  '.join(f'{c}={boxes[c]}' for c in CLASSES))
for a in sessions:
    for b in sessions:
        if a < b:
            assert not sessions[a] & sessions[b], f'phiên trùng giữa {a} và {b}'
print('OK: không phiên nào xuất hiện ở hai split')

In [ ]:
# Vẽ 6 ảnh train kèm box đúng
import matplotlib.pyplot as plt
from PIL import Image
from lab.viz import draw

fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for ax, r in zip(axes.flat, splits['train'][::230][:6]):
    ax.imshow(draw(Image.open('data/' + r['image_path']), r['objects']))
    ax.set_title(f"{r['image_id']} ({len(r['objects'])} box)"); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from lab.labels import build_prompt, objects_to_answer, objects_to_boxes, box_to_pixel, yolo_to_pixel

r = next(r for r in splits['train'] if len(r['objects']) >= 2)
print('PROMPT:', build_prompt('norm1000'))
print('ĐÁP ÁN:', objects_to_answer(r['objects']))

# Kiểm tra quy ngược: JSON 0–1000 → pixel phải trùng box gốc (sai số làm tròn ≤ 0,64 px theo x, 0,36 px theo y)
from lab.labels import yolo_to_box
err = 0.0
for row in splits['train']:
    for o in row['objects']:
        px = box_to_pixel(yolo_to_box(o['label'], o['yolo'])['bbox_2d'], 'norm1000', row['width'], row['height'])
        err = max(err, max(abs(a - c) for a, c in zip(px, yolo_to_pixel(o['yolo'], row['width'], row['height']))))
print(f'sai số quy ngược lớn nhất trên toàn train: {err:.2f} px')
pred_like = [{'label': b['label'], 'bbox': b['bbox_2d']} for b in objects_to_boxes(r['objects'])]
plt.figure(figsize=(9, 5)); plt.imshow(draw(Image.open('data/' + r['image_path']), r['objects'], pred_like))
plt.title('xanh = YOLO gốc, đỏ = JSON 0–1000 vẽ lại'); plt.axis('off'); plt.show()

## Kiến trúc Qwen3-VL-4B (câu 1)
Đếm tham số trên meta device: không tải trọng số, không tốn VRAM.

In [ ]:
from lab.vlm import describe_architecture
arch = describe_architecture()
for k, v in arch['params'].items():
    print(f'{k:32s} {v/1e6:9.1f} M')
print('module LoRA khớp regex:', arch['lora_modules'], '(kỳ vọng llm=252, llm+vit=348)')
print('tie_word_embeddings:', arch['tie_word_embeddings'])

## 3. E0: zero-shot

In [ ]:
!python -m lab.run predict --config configs/e0.yaml --out {OUT}/e0
!python -m lab.run score   --config configs/e0.yaml --out {OUT}/e0 --evidence report/evidence/e0.json

In [ ]:
# Câu 2: xem output thô và phân loại lỗi của E0
from scripts.score import parse_output
e0 = read_jsonl(f'{OUT}/e0/pred_test.jsonl')
for row in e0[:5]:
    print(row['image_id'], '→', repr(row['raw'][:300]), '\n')
stats = Counter()
for row in e0:
    raw, p = row['raw'], parse_output(row['raw'])
    stats['bọc ```json'] += '```' in raw
    if p is None:
        stats['JSON hỏng / sai schema'] += 1
        continue
    stats['trả []'] += not p
    stats['nhãn ngoài 3 lớp'] += sum(b['label'] not in CLASSES for b in p)
    stats['ảnh có toạ độ > 1000'] += any(max(b['bbox']) > 1000 for b in p)
    stats['tổng box dự đoán'] += len(p)
print(dict(stats))
print('nhãn dự đoán:', Counter(b['label'] for row in e0 for b in (parse_output(row['raw']) or [])).most_common(10))

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Chỉ cần processor (CPU), không nạp model.

In [ ]:
import torch
from transformers import AutoProcessor
from lab.vlm import make_example, collate, image_token_count
from lab.config import load_config

processor = AutoProcessor.from_pretrained('Qwen/Qwen3-VL-4B-Instruct')
cfg1 = load_config('configs/e1.yaml')
ex = make_example(processor, 'data', r, cfg1)
print({k: tuple(v.shape) for k, v in ex.items()})
print('số token ảnh:', image_token_count(processor, ex['input_ids']), '(kỳ vọng 220)')
ans_ids = ex['labels'][ex['labels'] != -100]
print('token tính loss:', len(ans_ids), '/', ex['input_ids'].shape[1])
print('giải mã phần có loss:', repr(processor.tokenizer.decode(ans_ids)))
b = collate([ex, make_example(processor, 'data', splits['train'][0], cfg1)], processor.tokenizer.pad_token_id)
print('batch:', {k: tuple(v.shape) for k, v in b.items()})
del processor

## 5. Setup LoRA / đóng băng cho E1, E2, E3
`lab/vlm.py: apply_setup()` đóng băng mọi tham số, gắn LoRA theo regex (252 module LLM; 348 khi thêm ViT),
với E2/E3 đưa `visual.merger` + `visual.deepstack_merger_list` lên fp32 và mở huấn luyện; in số module khớp
và số tham số huấn luyện (xem log của lệnh train bên dưới, và ô kiến trúc ở trên).

## 6. Vòng huấn luyện
`train()` dùng AdamW 2e-4, warmup 5% + cosine, GradScaler fp16, tích luỹ 4 × batch 2, clip 1.0,
lưu adapter + **projector riêng** (`extra_trainable.pt`) + checkpoint mỗi 100 bước, ghi `run.json`.

Mẹo: chạy E1 trước rồi gỡ lỗi trên 40 ảnh val trước khi chấm test.

In [ ]:
# E1: LoRA chỉ LLM
!python -m lab.run train   --config configs/e1.yaml --out {OUT}/e1 --resume
!python -m lab.run predict --config configs/e1.yaml --out {OUT}/e1 --split val --limit 40
!python -m lab.run score   --config configs/e1.yaml --out {OUT}/e1 --split val --limit 40

In [ ]:
# Nếu val ổn (F1 E1 >> E0, valid ≈ 1): chấm test MỘT lần
!python -m lab.run predict --config configs/e1.yaml --out {OUT}/e1
!python -m lab.run score   --config configs/e1.yaml --out {OUT}/e1 --evidence report/evidence/e1.json

In [ ]:
# E2: + projector (merger + deepstack_merger_list, fp32)
!python -m lab.run train   --config configs/e2.yaml --out {OUT}/e2 --resume
!python -m lab.run predict --config configs/e2.yaml --out {OUT}/e2
!python -m lab.run score   --config configs/e2.yaml --out {OUT}/e2 --evidence report/evidence/e2.json

In [ ]:
# E3: + LoRA trên ViT
!python -m lab.run train   --config configs/e3.yaml --out {OUT}/e3 --resume
!python -m lab.run predict --config configs/e3.yaml --out {OUT}/e3
!python -m lab.run score   --config configs/e3.yaml --out {OUT}/e3 --evidence report/evidence/e3.json

## 7. So sánh E0–E3

In [ ]:
# Đường loss + tài nguyên (câu 7)
fig, ax = plt.subplots(figsize=(9, 4))
for e in ('e1', 'e2', 'e3'):
    p = f'{OUT}/{e}/run.json'
    if os.path.exists(p):
        run = json.load(open(p))
        ax.plot(run['losses'], label=f"{e.upper()} ({run['trainable_params']/1e6:.1f}M tham số, "
                                      f"{run['peak_vram_gb']} GB, {run['train_minutes']} phút)")
ax.set_xlabel('bước'); ax.set_ylabel('loss'); ax.set_yscale('log'); ax.legend(); plt.show()
!python -m lab.summarize

In [ ]:
# Câu 3: cùng file dự đoán E1, chấm sai thang (coi output 0–1000 là pixel) → F1 thay đổi thế nào?
!python -m lab.run score --config configs/e1.yaml --out {OUT}/e1 --coord pixel --name E1-as-pixel --evidence work/e1_as_pixel.json

## 8. E4: ablation (đổi đúng MỘT biến so với setup tốt nhất)
Mặc định: `data_size` (10% vs 100% ảnh train). Giá trị 100% chính là setup gốc nên chỉ cần **một** lần train mới.
Các lựa chọn khác để ở dạng chú thích.

In [ ]:
BEST = 'e2'   # ← ĐỔI thành setup tốt nhất (e1/e2/e3) theo bảng ở trên
E4 = f'{OUT}/e4_data_size_0.1'
!python -m lab.run train   --config configs/{BEST}.yaml --set train_fraction=0.1 --set name=E4 --out {E4} --resume
!python -m lab.run predict --config configs/{BEST}.yaml --set train_fraction=0.1 --out {E4}
!python -m lab.run score   --config configs/{BEST}.yaml --set train_fraction=0.1 --out {E4} --name "E4 data_size=0.1" --evidence report/evidence/e4_data_size_0.1.json
# giá trị 1.0 = setup gốc: chấm lại đúng dự đoán test của BEST dưới tên E4
!python -m lab.run score   --config configs/{BEST}.yaml --out {OUT}/{BEST} --name "E4 data_size=1.0" --evidence report/evidence/e4_data_size_1.0.json

# --- Lựa chọn khác (chỉ chọn MỘT biến) ---
# rank: --set lora.r=4  (alpha tự = 2r)  và  --set lora.r=64  → e4_rank_4.json, e4_rank_64.json
# coord_scale: --set coord_scale=pixel (nhãn + prompt + chấm --coord pixel đều đổi theo) → e4_coord_scale_pixel.json

## Ảnh minh hoạ và phân tích lỗi theo lớp (câu 9)

In [ ]:
from lab.viz import make_figures, image_stats
make_figures(f'{OUT}/{BEST}/pred_test.jsonl', tag=BEST)          # 3 ảnh đúng + 3 ảnh sai
make_figures(f'{OUT}/e0/pred_test.jsonl', n_good=0, n_bad=2, tag='e0')
from IPython.display import Image as IPImage, display
for f in sorted(os.listdir('report/figures')):
    if f.endswith('.jpg'):
        print(f); display(IPImage('report/figures/' + f, width=640))

In [ ]:
# Recall theo lớp và theo kích thước box (diện tích tương đối) trên test cho setup tốt nhất
from scripts.score import _match, _to_norm1000, iou_xyxy
from lab.reward import gt_boxes
preds = {x['image_id']: x['raw'] for x in read_jsonl(f'{OUT}/{BEST}/pred_test.jsonl')}
buckets = Counter(); hits = Counter(); areas = {c: [] for c in CLASSES}
for e in splits['test']:
    p = parse_output(preds.get(e['image_id'])) or []
    for o, g in zip(e['objects'], gt_boxes(e['objects'])):
        a = o['yolo'][2] * o['yolo'][3]
        areas[o['label']].append(a)
        size = 'nhỏ <0.5%' if a < 0.005 else ('vừa <2%' if a < 0.02 else 'lớn ≥2%')
        buckets[(o['label'], size)] += 1
        hits[(o['label'], size)] += any(q['label'] == o['label'] and iou_xyxy(g['bbox'], q['bbox']) >= 0.5 for q in p)
import statistics
for c in CLASSES:
    print(f"{c:8s} median area={statistics.median(areas[c])*100:.2f}% | " + ' | '.join(
        f"{s}: {hits[(c, s)]}/{buckets[(c, s)]}" for s in ('nhỏ <0.5%', 'vừa <2%', 'lớn ≥2%')))

## Tổng hợp và kiểm tra bài nộp

In [ ]:
!python -m lab.summarize --write     # ghi bảng vào REPORT.md (giữa <!-- TABLE:START/END -->)
!python scripts/check_submission.py

## Lưu kết quả lên GitHub
Cần secret `GH_TOKEN` (GitHub token quyền `repo`). Chạy cuối mỗi phiên.

In [ ]:
from google.colab import userdata
tok = userdata.get('GH_TOKEN')
!git config user.name "Nguyen Van Quoc Viet" && git config user.email "legiongk123@gmail.com"
!git add report/ REPORT.md
!git commit -m "Kết quả thí nghiệm (evidence, ảnh, bảng)" || echo "không có gì mới"
!git push https://{tok}@github.com/vietvuivui/K4-Track-4-Day10-VisionLanguageModel.git HEAD:main

## 9. E5 (tuỳ chọn, +10): GRPO với reward IoU
Test reward đã có trong `lab/tests/test_lab_reward.py` (hoàn hảo, JSON hỏng, sai lớp, thừa box, rỗng, box khổng lồ).
`lab/grpo.py` viết theo API `GRPOTrainer` cho VLM; trl ghim 1.9.2, nếu tham số đổi tên hãy sửa theo docs đúng bản.

In [ ]:
!python -m pytest -q lab/tests/test_lab_reward.py
!python -m lab.grpo --base-config configs/{BEST}.yaml --base-out {OUT}/{BEST} --out {OUT}/e5
!python -m lab.run predict --config configs/{BEST}.yaml --out {OUT}/e5
!python -m lab.run score   --config configs/{BEST}.yaml --out {OUT}/e5 --name E5 --evidence report/evidence/e5.json

In [ ]:
# Reward hacking? reward theo bước + tỉ lệ output rỗng / box khổng lồ so với SFT
log = json.load(open(f'{OUT}/e5/grpo_log.json'))['log_history']
rk = next((k for k in log[0] if k.startswith('reward') and 'std' not in k), 'reward')
plt.plot([x.get(rk) for x in log if rk in x]); plt.xlabel('bước'); plt.ylabel(rk); plt.show()
for tag in (BEST, 'e5'):
    rows = [parse_output(x['raw']) for x in read_jsonl(f'{OUT}/{tag}/pred_test.jsonl')]
    ok = [p for p in rows if p is not None]
    big = sum(any((b['bbox'][2]-b['bbox'][0])*(b['bbox'][3]-b['bbox'][1]) > 250_000 for b in p) for p in ok)
    print(f"{tag}: rỗng={sum(not p for p in ok)}/{len(rows)}  box>25% ảnh={big}  "
          f"box/ảnh={sum(len(p) for p in ok)/max(len(ok),1):.2f}  JSON hỏng={len(rows)-len(ok)}")